# Model Training and Export


## 1. Import Libraries


In [1]:
import os
import json

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings

warnings.filterwarnings('ignore')
print('Libraries loaded successfully.')


Libraries loaded successfully.


## 2. Load Cleaned Dataset


In [2]:
df_model = pd.read_csv("../data/cleaned_dataset.csv")
print(f"Dataset: {df_model.shape[0]:,} rows x {df_model.shape[1]} columns")
df_model.head()


Dataset: 21,613 rows x 22 columns


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,lat,long,sqft_living15,sqft_lot15,sale_year,sale_month,house_age,was_renovated,years_since_renovation,total_rooms
0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,...,47.5112,-122.257,1340,5650,2014,10,59,0,59,4.00
1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,...,47.7210,-122.319,1690,7639,2014,12,63,1,23,5.25
2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,...,47.7379,-122.233,2720,8062,2015,2,82,0,82,3.00
3,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,...,47.5208,-122.393,1360,5000,2014,12,49,0,49,7.00
4,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,...,47.6168,-122.045,1800,7503,2015,2,28,0,28,5.00


## 3. Train-Test Split & Scaling


In [3]:
X = df_model.drop(columns=['price'])
y = df_model['price']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {X_train.shape[0]:,} samples")
print(f"Test set:  {X_test.shape[0]:,} samples")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

feature_columns = X.columns.tolist()
print(f"\nFeatures ({len(feature_columns)}): {feature_columns}")

Train set: 17,290 samples
Test set:  4,323 samples

Features (21): ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'lat', 'long', 'sqft_living15', 'sqft_lot15', 'sale_year', 'sale_month', 'house_age', 'was_renovated', 'years_since_renovation', 'total_rooms']


## 4. Training all Models


In [4]:
def evaluate_model(name, y_true, y_pred):
    """Compute regression metrics for a given model."""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {
        'Model': name,
        'MAE': round(mae, 2),
        'MSE': round(mse, 2),
        'RMSE': round(rmse, 2),
        'R2 Score': round(r2, 4)
    }

In [5]:
# Train all 5 models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'SVR': SVR(kernel='rbf')
}

results = []
for name, model_obj in models.items():
    model_obj.fit(X_train_scaled, y_train)
    preds = model_obj.predict(X_test_scaled)
    metrics = evaluate_model(name, y_test, preds)
    results.append(metrics)
    print(f"{name:25s} | R2 = {metrics['R2 Score']:.4f}")

results_df = pd.DataFrame(results)
results_df.sort_values(by='R2 Score', ascending=False)


Linear Regression         | R2 = 0.6972
Decision Tree             | R2 = 0.7170
Random Forest             | R2 = 0.8464
KNN                       | R2 = 0.7680
SVR                       | R2 = -0.0633


,Model,MAE,MSE,RMSE,R2 Score
2,Random Forest,73480.40,2.322691e+10,152403.77,0.8464
3,KNN,101350.56,3.506554e+10,187257.96,0.7680
1,Decision Tree,103464.45,4.278572e+10,206847.09,0.7170
0,Linear Regression,127647.57,4.577808e+10,213958.12,0.6972
4,SVR,229285.19,1.607447e+11,400929.84,-0.0633


## 5. Tune Random Forest with GridSearchCV


In [6]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [8, 12]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=1),
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")

Fitting 3 folds for each of 4 candidates, totalling 12 fits

Best Parameters: {'max_depth': 12, 'n_estimators': 200}


In [7]:
# Evaluate tuned model
best_rf_model = grid_search.best_estimator_
best_rf_preds = best_rf_model.predict(X_test_scaled)

tuned_metrics = evaluate_model('Random Forest (Tuned)', y_test, best_rf_preds)

# Final comparison table
final_results = pd.concat([results_df, pd.DataFrame([tuned_metrics])], ignore_index=True)
final_results = final_results.sort_values(by='R2 Score', ascending=False).reset_index(drop=True)
print("Final Model Comparison:")
final_results

Final Model Comparison:


,Model,MAE,MSE,RMSE,R2 Score
0,Random Forest (Tuned),76196.08,2.275109e+10,150834.65,0.8495
1,Random Forest,73480.40,2.322691e+10,152403.77,0.8464
2,KNN,101350.56,3.506554e+10,187257.96,0.7680
3,Decision Tree,103464.45,4.278572e+10,206847.09,0.7170
4,Linear Regression,127647.57,4.577808e+10,213958.12,0.6972
5,SVR,229285.19,1.607447e+11,400929.84,-0.0633


## 6. Export Model Artifacts for Flask API


In [8]:
# Output directory for model artifacts
OUTPUT_DIR = os.path.join('..', 'backend', 'model')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save trained model
model_path = os.path.join(OUTPUT_DIR, 'model.pkl')
joblib.dump(best_rf_model, model_path)
print(f"Model saved -> {model_path}")

# Save fitted scaler
scaler_path = os.path.join(OUTPUT_DIR, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"Scaler saved -> {scaler_path}")

# Save feature column order (important for prediction)
features_path = os.path.join(OUTPUT_DIR, 'feature_columns.json')
with open(features_path, 'w') as f:
    json.dump(feature_columns, f, indent=2)
print(f"Feature columns saved -> {features_path}")

print(f"\nAll artifacts exported to {OUTPUT_DIR}/")

Model saved -> ..\backend\model\model.pkl
Scaler saved -> ..\backend\model\scaler.pkl
Feature columns saved -> ..\backend\model\feature_columns.json

All artifacts exported to ..\backend\model/


## 7. Verify Exported Model


In [ ]:
# Quick check - load the exported model and predict on a sample
loaded_model = joblib.load(model_path)
loaded_scaler = joblib.load(scaler_path)

with open(features_path, 'r') as f:
    loaded_features = json.load(f)

# Sample house for testing
sample_house = {
    'bedrooms': 4,
    'bathrooms': 2.5,
    'sqft_living': 2400,
    'sqft_lot': 6000,
    'floors': 2.0,
    'waterfront': 0,
    'view': 0,
    'condition': 4,
    'grade': 8,
    'sqft_above': 2400,
    'sqft_basement': 0,
    'lat': 47.6062,
    'long': -122.3321,
    'sqft_living15': 2200,
    'sqft_lot15': 5800,
    'sale_year': 2015,
    'sale_month': 6,
    'house_age': 15,
    'was_renovated': 0,
    'years_since_renovation': 15,
    'total_rooms': 6.5
}

sample_df = pd.DataFrame([sample_house])[loaded_features]
sample_scaled = loaded_scaler.transform(sample_df)
predicted_price = loaded_model.predict(sample_scaled)[0]

print(f'Sample prediction: ${predicted_price:,.2f}')
print('\nModel export verified successfully!')


Sample prediction: $802,974.36

Model export verified successfully!
